# A4 Summary Sheet

Produces `summary_a4.pdf` — a one-page printable A4 with title, setup, the two sample-efficiency plots, per-algorithm observations and a key finding.

Drop this on the desk during your team review or your TA meeting.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
%cd /content/drive/MyDrive/imitation_learning

In [ ]:
import os, csv
from collections import defaultdict
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

ALGOS = ['iqlearn', 'csil', 'csilsoar']
ALGO_LABEL = {'iqlearn': 'IQ-Learn', 'csil': 'CSIL', 'csilsoar': 'CSIL+SOAR'}
ALGO_COLOR = {'iqlearn': '#1f77b4', 'csil': '#ff7f0e', 'csilsoar': '#2ca02c'}
ENVS = ['CartPole', 'Pendulum']
K_VALUES = [1, 3, 5, 10, 15]
SEEDS = [42, 43, 44]

def load_one(path):
    steps, rewards = [], []
    with open(path) as f:
        for row in csv.DictReader(f):
            steps.append(int(row['step']))
            rewards.append(float(row['eval_reward']))
    return np.array(steps), np.array(rewards)

def candidate_paths(algo, env, K, seed):
    return [f'logs/{algo}_{env}{suffix}_K{K}_seed{seed}.csv' for suffix in ['', '-v1']]

data = defaultdict(lambda: defaultdict(lambda: defaultdict(dict)))
for algo in ALGOS:
    for env in ENVS:
        for K in K_VALUES:
            for seed in SEEDS:
                for p in candidate_paths(algo, env, K, seed):
                    if os.path.exists(p):
                        data[algo][env][K][seed] = load_one(p)
                        break
print('Data loaded.')

## Build the A4 page

In [ ]:
# A4 portrait dimensions in inches
A4_W, A4_H = 8.27, 11.69

fig = plt.figure(figsize=(A4_W, A4_H))
gs = GridSpec(
    nrows=10, ncols=2,
    height_ratios=[0.6, 0.5, 0.4, 3.4, 0.3, 1.6, 1.6, 1.6, 1.4, 0.3],
    hspace=0.35, wspace=0.25,
    left=0.07, right=0.95, top=0.96, bottom=0.03,
)

# ─── Title ────────────────────────────────────────────────
ax_title = fig.add_subplot(gs[0, :]); ax_title.axis('off')
ax_title.text(0.5, 0.7, 'Imitation Learning — IQ-Learn vs CSIL vs CSIL+SOAR',
              ha='center', va='center', fontsize=16, fontweight='bold')
ax_title.text(0.5, 0.15, 'EE-568 Applied Project 3  ·  Louis Grange, Raki Ben Mustapha, Daniel Ataide',
              ha='center', va='center', fontsize=10, style='italic', color='#555')

# ─── Setup block ───────────────────────────────────────────
ax_setup = fig.add_subplot(gs[1, :]); ax_setup.axis('off')
setup_txt = (
    'Setup:  SAC experts → K ∈ {1, 3, 5, 10, 15} trajectories per env  ·  '
    '3 algorithms × 2 envs × 5 K × 3 seeds = 90 runs total\n'
    'Envs:   CartPole-v1 (discrete, max reward = 500)   ·   '
    'Pendulum-v1 (continuous, expert ≈ −200, random ≈ −1500)'
)
ax_setup.text(0.0, 1.0, setup_txt, ha='left', va='top', fontsize=9,
              family='monospace', color='#222')

# ─── Section header: Results ───────────────────────────────
ax_h1 = fig.add_subplot(gs[2, :]); ax_h1.axis('off')
ax_h1.text(0.0, 0.5, '▶ Sample efficiency  (median over 3 seeds, IQR error bars)',
           ha='left', va='center', fontsize=12, fontweight='bold')

# ─── The two sample-efficiency plots ───────────────────────
def max_per_seed(d): return [v[1].max() for v in d.values()]

for col, env in enumerate(ENVS):
    ax = fig.add_subplot(gs[3, col])
    for algo in ALGOS:
        xs, med, p25, p75 = [], [], [], []
        for K in K_VALUES:
            m = max_per_seed(data[algo][env][K])
            if not m: continue
            xs.append(K); med.append(np.median(m))
            p25.append(np.percentile(m, 25)); p75.append(np.percentile(m, 75))
        if not xs: continue
        med, p25, p75 = map(np.array, [med, p25, p75])
        ax.errorbar(xs, med, yerr=[med - p25, p75 - med],
                    color=ALGO_COLOR[algo], label=ALGO_LABEL[algo],
                    marker='o', capsize=3, linewidth=1.8)
    ax.set_xscale('log'); ax.set_xticks(K_VALUES); ax.set_xticklabels(K_VALUES)
    ax.set_xlabel('K (expert trajectories)', fontsize=9)
    ax.set_ylabel('Best reward (median)', fontsize=9)
    ax.set_title(f'{env}-v1', fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8, loc='best', frameon=True)
    ax.tick_params(labelsize=8)

# ─── Section header: Observations ──────────────────────────
ax_h2 = fig.add_subplot(gs[4, :]); ax_h2.axis('off')
ax_h2.text(0.0, 0.3, '▶ Observations per algorithm',
           ha='left', va='center', fontsize=12, fontweight='bold')

# ─── Observations text blocks ──────────────────────────────
def obs_block(ax, color, title, lines):
    ax.axis('off')
    ax.text(0.0, 1.0, title, ha='left', va='top',
            fontsize=10.5, fontweight='bold', color=color)
    body = '\n'.join(lines)
    ax.text(0.02, 0.78, body, ha='left', va='top', fontsize=9, color='#222')

ax_iq = fig.add_subplot(gs[5, :])
obs_block(ax_iq, ALGO_COLOR['iqlearn'], 'IQ-Learn  (this work)', [
    '✓ Solved CartPole reliably at K≥3 (max reward = 500/500).',
    '✓ Reached expert level on Pendulum at K=10 (median best ≈ −150, vs expert ≈ −200).',
    '✗ Required substantial tuning to stabilize on continuous control:',
    '      value-mode loss baseline (vs initial-state-only),  gradient clipping (max-norm 10/1),',
    '      auto-tuned entropy α, action scaling fix on the actor→critic path.',
])

ax_cs = fig.add_subplot(gs[6, :])
obs_block(ax_cs, ALGO_COLOR['csil'], 'CSIL', [
    '✓ Solved CartPole at K≥3 (median 500) — BC pre-training alone is sufficient.',
    '✗ Failed to solve Pendulum (median best ≈ −574 to −1116 depending on K).',
    '✗ Non-monotonic in K: best at K=1, K=3; worse at K=10. Likely cause: BC pre-training',
    '      overfits to expert states → confidently wrong policy on out-of-distribution states.',
])

ax_so = fig.add_subplot(gs[7, :])
obs_block(ax_so, ALGO_COLOR['csilsoar'], 'CSIL + SOAR (4-critic ensemble, optimism β=1)', [
    '≈ Matches CSIL on CartPole (both reach 500 for K≥3).',
    '≈ Similar to CSIL on Pendulum — optimistic critic bonus did not help.',
    '↳ Consistent with last year\'s project poster (f-IRL+SOAR also failed to improve over f-IRL).',
])

# ─── Key finding ───────────────────────────────────────────
ax_key = fig.add_subplot(gs[8, :])
ax_key.axis('off')
ax_key.text(0.0, 1.0, '▶ Key finding', ha='left', va='top',
            fontsize=12, fontweight='bold')
ax_key.text(0.0, 0.78,
    'On Pendulum at K=10:  IQ-Learn median best ≈ −150  vs  CSIL ≈ −1116  →  ~9× gap.\n'
    'At low K (1, 3) CSIL is slightly better; from K=5 onward IQ-Learn dominates dramatically.\n'
    'Pure inverse soft-Q learning scales more effectively with K than BC-initialized methods\n'
    'in the scarce-data continuous-control regime tested here.',
    ha='left', va='top', fontsize=9.5, color='#222')

fig.savefig('summary_a4.pdf')
fig.savefig('summary_a4.png', dpi=200)
print('Saved summary_a4.pdf and summary_a4.png')
plt.show()

## Download the PDF

In [ ]:
from google.colab import files
files.download('summary_a4.pdf')